# Evaluation of Inference vs. SFT Adapter

## Goal
Create a simple and readable comparison between:
- the base inference model output
- the SFT adapter model output
- the gold answers

The notebook loads all three CSV files, merges them by ID, computes simple matching metrics, and saves:
- one detailed comparison file
- one summary file

In [ ]:
# Import the required libraries
import pandas as pd
from difflib import SequenceMatcher

## 1. Configuration

This section defines all input and output file paths.

In [ ]:
# Input files
GOLD_PATH = "./data/dataset_prompt_answers_clean.csv"
INFERENCE_PATH = "./outputs/inference_results_local_8bit.csv"
ADAPTER_PATH = "./outputs/inference_results_local_8bit_sft_adapter.csv"

# Output files
DETAIL_OUT_PATH = "./outputs/eval_detailed_comparison.csv"
SUMMARY_OUT_PATH = "./outputs/eval_summary.csv" 

## 2. Load the CSV Files

All three CSV files are loaded into pandas Dataframes

Each Dataframe is a table:
- gold_df contains the reference answers
- inference_df contains the base model predictions
- adapter_df contains the adapter model predictions

In [ ]:
# Load all CSV files
gold_df = pd.read_csv(GOLD_PATH, encoding="utf-8-sig")
inference_df = pd.read_csv(INFERENCE_PATH, encoding="utf-8-sig")
adapter_df = pd.read_csv(ADAPTER_PATH, encoding="utf-8-sig")

## 3. Keep Only the Needed Columns

Only the relevant columns are kept so the comparison stays clean and easy to follow.

The answer columns are then renamed so both model outputs can be clearly distinguished later.

In [ ]:
# Keep only the relevant columns from the gold file
gold_df = gold_df[["id", "prompt", "correct_answer"]].copy()
gold_df.columns = ["ID", "Prompt", "Gold_Answer"]

# Keep only the relevant columns from the inference result file
inference_df = inference_df[["ID", "Answer"]].copy()
inference_df.columns = ["ID", "Inference_Answer"]

# Keep only the relevant columns from the adapter result file
adapter_df = adapter_df[["ID", "Answer"]].copy()
adapter_df.columns = ["ID", "Adapter_Answer"]

# Convert all IDs to strings to avoid merge issues later
gold_df["ID"] = gold_df["ID"].astype(str)
inference_df["ID"] = inference_df["ID"].astype(str)
adapter_df["ID"] = adapter_df["ID"].astype(str)

## 4. Merge the Tables

The three tables are joined using the ID column.

After the merge, each row contains:
- the original prompt
- the gold answer
- the inference answer
- the adapter answer

Missing answers are replaced with empty strings so later comparisons do not fail.

In [ ]:
# Merge gold answers with inference answers
df = gold_df.merge(inference_df, on="ID", how="left")

# Merge the result with adapter answers
df = df.merge(adapter_df, on="ID", how="left")

# Replace missing values with empty strings
df["Inference_Answer"] = df["Inference_Answer"].fillna("").astype(str)
df["Adapter_Answer"] = df["Adapter_Answer"].fillna("").astype(str)
df["Gold_Answer"] = df["Gold_Answer"].fillna("").astype(str)

## 5. Text Cleaning for Fairer Comparison

The answers are cleaned in a very simple way before comparison:
- convert to lowercase
- remove leading and trailing spaces
- collapse repeated spaces into one space

This helps avoid unfair mismatches caused only by formatting differences.

In [ ]:
# Create cleaned versions of all answer columns
df["Gold_Clean"] = (df["Gold_Answer"].str.lower().str.strip().str.replace(r"\s+", " ", regex=True))

df["Inference_Clean"] = (df["Inference_Answer"].str.lower().str.strip().str.replace(r"\s+", " ", regex=True))

df["Adapter_Clean"] = (df["Adapter_Answer"].str.lower().str.strip().str.replace(r"\s+", " ", regex=True))

## 6. Exact Match

Exact match checks whether two answers are identical.

Two versions are calculated:
- raw exact match
- cleaned exact match

Raw exact match is very strict.
Cleaned exact match is still strict, but ignores simple formatting differences.

In [ ]:
# Exact match on the original text
df["Inference_Exact_Match"] = (df["Inference_Answer"] == df["Gold_Answer"]).astype(int)
df["Adapter_Exact_Match"] = (df["Adapter_Answer"] == df["Gold_Answer"]).astype(int)

# Exact match on the cleaned text
df["Inference_Clean_Exact_Match"] = (df["Inference_Clean"] == df["Gold_Clean"]).astype(int)
df["Adapter_Clean_Exact_Match"] = (df["Adapter_Clean"] == df["Gold_Clean"]).astype(int)

## 7. Similarity Score

A second metric is added because exact match is often too strict for free-text answers.

SequenceMatcher returns a similarity value between 0 and 1:
- 1 means the texts are identical
- values closer to 0 mean the texts are more different

This does not measure legal correctness perfectly, but it gives a quick baseline comparison.

In [ ]:
# Lists for similarity scores
inference_scores = []
adapter_scores = []

# Compute one similarity score per row
for _, row in df.iterrows():
    gold_text = row["Gold_Clean"]
    inference_text = row["Inference_Clean"]
    adapter_text = row["Adapter_Clean"]

    inference_score = SequenceMatcher(None, gold_text, inference_text).ratio()
    adapter_score = SequenceMatcher(None, gold_text, adapter_text).ratio()

    inference_scores.append(inference_score)
    adapter_scores.append(adapter_score)

# Store similarity scores in the table
df["Inference_Similarity"] = inference_scores
df["Adapter_Similarity"] = adapter_scores

## 8. Winner Per Row

For each row, the model with the higher similarity score is marked as the winner.

Possible values:
- Inference
- Adapter
- Tie

In [ ]:
# Determine which model is closer to the gold answer for each row
winner_list = []

for _, row in df.iterrows():
    inf_score = row["Inference_Similarity"]
    adp_score = row["Adapter_Similarity"]

    if adp_score > inf_score:
        winner_list.append("Adapter")
    elif inf_score > adp_score:
        winner_list.append("Inference")
    else:
        winner_list.append("Tie")

df["Winner"] = winner_list

## 9. Summary Metrics

This section creates one compact summary table for both models.

The summary includes:
- number of rows
- exact match rate
- cleaned exact match rate
- average similarity
- number of row wins

In [ ]:
# Build the summary rows for both models
summary_rows = []

summary_rows.append({
    "Model": "Inference",
    "Rows": len(df),
    "Exact_Match_Rate": df["Inference_Exact_Match"].mean(),
    "Clean_Exact_Match_Rate": df["Inference_Clean_Exact_Match"].mean(),
    "Average_Similarity": df["Inference_Similarity"].mean(),
    "Wins": (df["Winner"] == "Inference").sum()
})

summary_rows.append({
    "Model": "Adapter",
    "Rows": len(df),
    "Exact_Match_Rate": df["Adapter_Exact_Match"].mean(),
    "Clean_Exact_Match_Rate": df["Adapter_Clean_Exact_Match"].mean(),
    "Average_Similarity": df["Adapter_Similarity"].mean(),
    "Wins": (df["Winner"] == "Adapter").sum()
})

summary_df = pd.DataFrame(summary_rows)
summary_df

## 10. Save the Results

Two files are written:
- a detailed file with all rows and all metrics
- a summary file with the compact model comparison

In [ ]:
# Save the detailed row-level comparison
df.to_csv(DETAIL_OUT_PATH, index=False, encoding="utf-8-sig")

# Save the compact summary table
summary_df.to_csv(SUMMARY_OUT_PATH, index=False, encoding="utf-8-sig")

## 11. Final Output

The final printout gives:
- the summary table
- the number of tie rows
- the locations of the saved files

In [ ]:
print()
print("Summary:")
print(summary_df)
print()

print("Ties:", (df["Winner"] == "Tie").sum())